确认WGFMU模块调用正常，确认WGFMU通道接口，确认是否能正常返回值
预留关键参数接口如：测量模式、循环次数、量程、平均/积分、采样间隔等

WGFMU 的官方定位为：

生成任意线性波形

同时做高速 I/V 测量

支持固定量程下的受控量程切换

支持可变采样率和硬件 averaging

本文件目标：
1. 确认 B1530A WGFMU instrument library 链路可用
2. 确认实际可用的 WGFMU 通道号
3. 确认最小单通道波形可以成功下发
4. 确认最小测量事件可以成功回读
5. 保存 setup/debug 文件，作为后续 11/12 的基线


In [ ]:
# 导入包，WGFMU 官方是 instrument library，不是纯 SCPI，所以这个 notebook 假设后面
# 会有一个 Python 封装层去调用官方库。官方手册也明确说，WGFMU 是通过 instrument library 控制的。
from __future__ import annotations

import json
import time
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Optional

import yaml
import pandas as pd

from fefetlab.instruments.visa_session import VisaConfig

In [ ]:
# 读取配置文件，准备 VISA 配置和 channel map
ROOT = Path.cwd().resolve()
CFG_DIR = ROOT / "configs"

with open(CFG_DIR / "instruments.yaml", "r", encoding="utf-8") as f:
    inst_cfg = yaml.safe_load(f)["b1500"]

with open(CFG_DIR / "channel_map.yaml", "r", encoding="utf-8") as f:
    channel_cfg = yaml.safe_load(f)

visa_cfg = VisaConfig(
    resource=inst_cfg["resource"],
    timeout_ms=inst_cfg["timeout_ms"],
    write_termination=inst_cfg["write_termination"],
    read_termination=inst_cfg["read_termination"],
    send_end=inst_cfg["send_end"],
)

print("B1500 resource =", visa_cfg.resource)
print("timeout_ms =", visa_cfg.timeout_ms)
print("channel_map top keys =", list(channel_cfg.keys()))

In [ ]:
#WGFMU 参数 dataclass
#这里按照官方执行流程，把参数拆成三层更清楚：
#PulsePatternParams：波形
#MeasureEventParams：测量事件
#WgfmuSmokeConfig：运行配置
@dataclass
class PulsePatternParams:
    chan_id: int
    pattern_name: str
    v_init: float
    v_pulse: float
    t_rise_s: float
    t_high_s: float
    t_fall_s: float
    t_base_s: float
    repeat_count: int = 1


@dataclass
class MeasureEventParams:
    event_name: str
    start_time_s: float
    points: int
    interval_s: float
    average_s: float
    raw_data_mode: str = "averaged"   # 先只做 averaged


@dataclass
class WgfmuSmokeConfig:
    label: str
    operation_mode: str = "FASTIV"    # FASTIV / PG / DC
    force_voltage_range: str = "AUTO"
    measure_mode: str = "CURRENT"     # CURRENT / VOLTAGE
    measure_current_range: str = "1MA"
    measure_voltage_range: str = "10V"
    measure_enabled: bool = True
    treat_warning_as_error: bool = False
    timeout_s: float = 30.0
    notes: str = ""


In [ ]:
# 这里先假设用 101 做 smoke
# 真正运行前，会在后面的“发现通道”cell里确认

pulse_cfg = PulsePatternParams(
    chan_id=101,
    pattern_name="smoke_pulse",
    v_init=0.0,
    v_pulse=-1.0,
    t_rise_s=1e-6,
    t_high_s=2e-6,
    t_fall_s=1e-6,
    t_base_s=2e-6,
    repeat_count=1,
)

meas_cfg = MeasureEventParams(
    event_name="Imeas",
    start_time_s=1.5e-6,
    points=20,
    interval_s=200e-9,
    average_s=100e-9,
    raw_data_mode="averaged",
)

run_cfg = WgfmuSmokeConfig(
    label="10_wgfmu_smoke_single_channel",
    operation_mode="FASTIV",
    force_voltage_range="AUTO",
    measure_mode="CURRENT",
    measure_current_range="1MA",
    measure_voltage_range="10V",
    measure_enabled=True,
    treat_warning_as_error=False,
    timeout_s=30.0,
    notes="单通道最小脉冲+最小测量事件 smoke"
)

print(asdict(pulse_cfg))
print(asdict(meas_cfg))
print(asdict(run_cfg))

In [ ]:
# 定义一个“Python 到 WGFMU 官方库”的包装接口
class WgfmuLib:
    """
    Python 对官方 WGFMU Instrument Library 的最小包装接口。
    这里先定义 notebook 需要的 API 形状。
    后面你可以用 ctypes / cffi / pybind11 去接官方库。
    """

    def __init__(self):
        self.session_opened = False

    # ---------- session ----------
    def open_session(self, resource: str):
        raise NotImplementedError

    def close_session(self):
        raise NotImplementedError

    def initialize(self):
        raise NotImplementedError

    def clear(self):
        raise NotImplementedError

    def set_timeout(self, timeout_s: float):
        raise NotImplementedError

    def get_channel_ids(self) -> list[int]:
        raise NotImplementedError

    def get_error_summary(self) -> str:
        raise NotImplementedError

    def get_warning_summary(self) -> str:
        raise NotImplementedError

    def treat_warnings_as_errors(self, level: str):
        raise NotImplementedError

    # ---------- offline setup ----------
    def create_pattern(self, pattern: str, init_v: float):
        raise NotImplementedError

    def add_vector(self, pattern: str, dtime_s: float, voltage: float):
        raise NotImplementedError

    def set_measure_event(
        self,
        pattern: str,
        event: str,
        time_s: float,
        points: int,
        interval_s: float,
        average_s: float,
        raw_data_mode: str,
    ):
        raise NotImplementedError

    def add_sequence(self, chan_id: int, pattern: str, count: int):
        raise NotImplementedError

    def export_ascii(self, filepath: str):
        raise NotImplementedError

    # ---------- online setup ----------
    def set_operation_mode(self, chan_id: int, mode: str):
        raise NotImplementedError

    def set_force_voltage_range(self, chan_id: int, rng: str):
        raise NotImplementedError

    def set_measure_enabled(self, chan_id: int, enabled: bool):
        raise NotImplementedError

    def set_measure_mode(self, chan_id: int, mode: str):
        raise NotImplementedError

    def set_measure_current_range(self, chan_id: int, rng: str):
        raise NotImplementedError

    def set_measure_voltage_range(self, chan_id: int, rng: str):
        raise NotImplementedError

    def connect(self, chan_id: int):
        raise NotImplementedError

    def disconnect(self, chan_id: int):
        raise NotImplementedError

    def execute(self):
        raise NotImplementedError

    def wait_until_completed(self):
        raise NotImplementedError

    # ---------- result ----------
    def get_measure_value_size(self, chan_id: int) -> tuple[int, int]:
        raise NotImplementedError

    def get_measure_values(self, chan_id: int) -> pd.DataFrame:
        raise NotImplementedError

In [ ]:
# 写一个占位后端，先让 notebook 能跑起来，后面再替换成真正调用官方库的实现
class DummyWgfmuLib(WgfmuLib):
    def __init__(self):
        super().__init__()
        self._channels = [101, 102]
        self._connected = set()

    def open_session(self, resource: str):
        self.session_opened = True

    def close_session(self):
        self.session_opened = False

    def initialize(self):
        return 0

    def clear(self):
        return 0

    def set_timeout(self, timeout_s: float):
        return 0

    def get_channel_ids(self) -> list[int]:
        return self._channels

    def get_error_summary(self) -> str:
        return ""

    def get_warning_summary(self) -> str:
        return ""

    def treat_warnings_as_errors(self, level: str):
        return 0

    def create_pattern(self, pattern: str, init_v: float):
        return 0

    def add_vector(self, pattern: str, dtime_s: float, voltage: float):
        return 0

    def set_measure_event(self, pattern, event, time_s, points, interval_s, average_s, raw_data_mode):
        return 0

    def add_sequence(self, chan_id: int, pattern: str, count: int):
        return 0

    def export_ascii(self, filepath: str):
        Path(filepath).write_text("dummy export", encoding="utf-8")

    def set_operation_mode(self, chan_id: int, mode: str):
        return 0

    def set_force_voltage_range(self, chan_id: int, rng: str):
        return 0

    def set_measure_enabled(self, chan_id: int, enabled: bool):
        return 0

    def set_measure_mode(self, chan_id: int, mode: str):
        return 0

    def set_measure_current_range(self, chan_id: int, rng: str):
        return 0

    def set_measure_voltage_range(self, chan_id: int, rng: str):
        return 0

    def connect(self, chan_id: int):
        self._connected.add(chan_id)
        return 0

    def disconnect(self, chan_id: int):
        self._connected.discard(chan_id)
        return 0

    def execute(self):
        return 0

    def wait_until_completed(self):
        return 0

    def get_measure_value_size(self, chan_id: int) -> tuple[int, int]:
        return 20, 20

    def get_measure_values(self, chan_id: int) -> pd.DataFrame:
        import numpy as np
        t = np.arange(20) * 200e-9
        i = np.linspace(1e-9, 3e-6, 20)
        return pd.DataFrame({"time_s": t, "value": i})

In [ ]:
#实例化占位后端，测试接口形状，先用 dummy，后面改成真实的 RealWgfmuLib()
wg = DummyWgfmuLib()
print(type(wg).__name__)

In [ ]:
# 建立本次运行目录
run_dir = ROOT / "runs" / time.strftime("%Y%m%d_%H%M%S_10_wgfmu_smoke")
run_dir.mkdir(parents=True, exist_ok=True)

paths = {
    "meta": run_dir / "meta.json",
    "export_ascii": run_dir / "setup_export.csv",
    "parsed": run_dir / "parsed.csv",
    "qc": run_dir / "qc.csv",
}

paths

In [ ]:
# 通道发现
wg.open_session(visa_cfg.resource)
wg.set_timeout(run_cfg.timeout_s)

channel_ids = wg.get_channel_ids()
print("Detected WGFMU channel ids:", channel_ids)

# 如果你配置里的 chan_id 不在实际发现列表里，直接报错
if pulse_cfg.chan_id not in channel_ids:
    raise RuntimeError(
        f"pulse_cfg.chan_id={pulse_cfg.chan_id} 不在检测到的 WGFMU 通道列表 {channel_ids} 中"
    )

In [ ]:
# 离线阶段
wg.clear()

# pattern
wg.create_pattern(pulse_cfg.pattern_name, pulse_cfg.v_init)
wg.add_vector(pulse_cfg.pattern_name, pulse_cfg.t_rise_s, pulse_cfg.v_pulse)
wg.add_vector(pulse_cfg.pattern_name, pulse_cfg.t_high_s, pulse_cfg.v_pulse)
wg.add_vector(pulse_cfg.pattern_name, pulse_cfg.t_fall_s, pulse_cfg.v_init)
wg.add_vector(pulse_cfg.pattern_name, pulse_cfg.t_base_s, pulse_cfg.v_init)

# measurement event
wg.set_measure_event(
    pattern=pulse_cfg.pattern_name,
    event=meas_cfg.event_name,
    time_s=meas_cfg.start_time_s,
    points=meas_cfg.points,
    interval_s=meas_cfg.interval_s,
    average_s=meas_cfg.average_s,
    raw_data_mode=meas_cfg.raw_data_mode,
)

# sequence
wg.add_sequence(
    chan_id=pulse_cfg.chan_id,
    pattern=pulse_cfg.pattern_name,
    count=pulse_cfg.repeat_count,
)

# 导出 setup 供检查
wg.export_ascii(str(paths["export_ascii"]))

print("Offline setup done.")
print("Exported setup file ->", paths["export_ascii"])

In [ ]:
# 在现阶段
wg.initialize()

if run_cfg.treat_warning_as_error:
    wg.treat_warnings_as_errors("SEVERE")

wg.set_operation_mode(pulse_cfg.chan_id, run_cfg.operation_mode)
wg.set_force_voltage_range(pulse_cfg.chan_id, run_cfg.force_voltage_range)
wg.set_measure_enabled(pulse_cfg.chan_id, run_cfg.measure_enabled)
wg.set_measure_mode(pulse_cfg.chan_id, run_cfg.measure_mode)

if run_cfg.measure_mode.upper() == "CURRENT":
    wg.set_measure_current_range(pulse_cfg.chan_id, run_cfg.measure_current_range)
else:
    wg.set_measure_voltage_range(pulse_cfg.chan_id, run_cfg.measure_voltage_range)

wg.connect(pulse_cfg.chan_id)

print("Online setup done.")

In [ ]:
# 执行smoke
t0 = time.time()

wg.execute()
wg.wait_until_completed()

t1 = time.time()
print(f"Execution completed in {t1 - t0:.3f} s")

In [ ]:
# 读取结果
complete, total = wg.get_measure_value_size(pulse_cfg.chan_id)
print("measure values:", complete, "/", total)

df = wg.get_measure_values(pulse_cfg.chan_id).copy()

# 统一列名
if "value" in df.columns:
    if run_cfg.measure_mode.upper() == "CURRENT":
        df = df.rename(columns={"value": "current_A"})
    else:
        df = df.rename(columns={"value": "voltage_V"})

display(df.head())
print("n_rows =", len(df))

In [ ]:
# 基础QC，看链路是否通
issues = []

if len(df) == 0:
    issues.append("empty_dataframe")

if complete != total:
    issues.append(f"incomplete_measurement:{complete}/{total}")

if run_cfg.measure_mode.upper() == "CURRENT" and "current_A" not in df.columns:
    issues.append("missing_current_A")

if run_cfg.measure_mode.upper() == "VOLTAGE" and "voltage_V" not in df.columns:
    issues.append("missing_voltage_V")

qc_df = pd.DataFrame([{
    "label": run_cfg.label,
    "status": "ok" if not issues else "suspect",
    "issues": ";".join(issues),
    "n_rows": len(df),
    "complete": complete,
    "total": total,
}])

display(qc_df)

In [ ]:
# 保存数据
meta = {
    "pulse_cfg": asdict(pulse_cfg),
    "meas_cfg": asdict(meas_cfg),
    "run_cfg": asdict(run_cfg),
    "detected_channel_ids": channel_ids,
    "complete": complete,
    "total": total,
    "issues": issues,
    "saved_at": time.strftime("%Y-%m-%d %H:%M:%S"),
}

df.to_csv(paths["parsed"], index=False, encoding="utf-8-sig")
qc_df.to_csv(paths["qc"], index=False, encoding="utf-8-sig")

with open(paths["meta"], "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print("saved:")
for k, v in paths.items():
    print(f"  {k}: {v}")

In [ ]:
# 收尾
try:
    wg.disconnect(pulse_cfg.chan_id)
finally:
    wg.close_session()

print("WGFMU session closed safely.")